### NDVI LMM


VARIABLE ROLES:
  - MAIN PREDICTORS: Built-form characteristics (reldiff_built_up_ratio,
                     reldiff_avg_building_height, reldiff_avg_building_footprint,
                     diff_res_subclass_share_mfh_ab)

  - CONTROL VARIABLES (CONFOUNDERS):
    * Temporal: years_since_construction_start
    * Spatial: latitude, longitude (geographic gradients)
    * Categorical: urban-rural classification, land-use type

Standardization: Z-score transformation (X - mean(X)) / std(X)
This allows a direct comparison of effect sizes between variables


In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from statsmodels.formula.api import mixedlm
import geopandas as gpd

warnings.filterwarnings('ignore', category=FutureWarning)


# ============================================================================
# Step 1: Load and prepare data
# ============================================================================

data_path = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2\analysis_dataset.csv"
df = pd.read_csv(data_path)

outcome_var = 'difference_NDVI'

# ============================================================================
# CONCEPTUAL DISTINCTION: BUILT-FORM vs CONTROLS
# ============================================================================

# Variables of substantive interest: BUILT-FORM CHARACTERISTICS
built_form_vars = [
    'reldiff_built_up_ratio',
    'reldiff_avg_building_height',
    'reldiff_avg_building_footprint',
    'diff_res_subclass_share_mfh_ab'
]

# Control variables: TEMPORAL (Note: no environmental control, NDVI is the outcome)
temporal_control_vars = [
    'years_since_construction_start'
]

# Control variables: SPATIAL (broad geographic gradients)
spatial_control_vars = [
    'lat',
    'lon'
]

# Control variables: CONTEXTUAL
categorical_control_vars = [
    'nhda_degurba_code',
    'nhda_sur_class_2021_short'
]

# Build complete fixed effects list (order: substantive, then controls)
fixed_vars = built_form_vars + temporal_control_vars + spatial_control_vars

random_intercept_var = 'nhda_id'

model_vars = [outcome_var] + fixed_vars + categorical_control_vars + [random_intercept_var]

# Prepare data
df_model = df[model_vars].copy().dropna()
df_model = df_model.reset_index(drop=True)

# ============================================================================
# Step 2: Outlier detection
# ============================================================================

# NOTE: Outlier detection excludes spatial_control_vars (lat, lon)
outlier_detection_vars = [outcome_var] + built_form_vars + temporal_control_vars

z_scores = pd.DataFrame()
for var in outlier_detection_vars:
    z_scores[var] = np.abs(stats.zscore(df_model[var]))

outlier_mask = (z_scores.fillna(0) > 3).any(axis=1).values.astype(bool)
n_outliers = outlier_mask.sum()

# ============================================================================
# Step 3: Standardization
# ============================================================================

df_standardized = df_model.copy()
df_unstandardized = df_model.copy()

original_means = {}
original_stds = {}

# Standardize ALL continuous variables (including spatial controls)
for var in fixed_vars:
    original_means[var] = df_model[var].mean()
    original_stds[var] = df_model[var].std()

    # Standardize: (X - mean) / std
    df_standardized[var] = (df_model[var] - original_means[var]) / original_stds[var]

# Set reference categories
for df_temp in [df_standardized, df_unstandardized]:
    df_temp['nhda_degurba_code'] = df_temp['nhda_degurba_code'].astype('category')
    df_temp['nhda_sur_class_2021_short'] = df_temp['nhda_sur_class_2021_short'].astype('category')

# ============================================================================
# Step 4: Split with/without outliers
# ============================================================================

# Create outlier flag
df_standardized['has_outlier'] = outlier_mask
df_unstandardized['has_outlier'] = outlier_mask

# Split
df_std_with = df_standardized.copy()
df_std_without = df_standardized[~outlier_mask].copy()
df_unstd_with = df_unstandardized.copy()
df_unstd_without = df_unstandardized[~outlier_mask].copy()

for df_temp in [df_std_with, df_std_without, df_unstd_with, df_unstd_without]:
    df_temp = df_temp.drop(columns=['has_outlier'], errors='ignore').reset_index(drop=True)

# ============================================================================
# Step 5: Build model formula
# ============================================================================

formula = (
    f"{outcome_var} ~ "
    "reldiff_built_up_ratio + "
    "reldiff_avg_building_height + "
    "reldiff_avg_building_footprint + "
    "diff_res_subclass_share_mfh_ab + "
    "years_since_construction_start + "  # Temporal control
    "lat + lon + "  # Spatial controls
    "C(nhda_degurba_code) + "  # Contextual controls
    "C(nhda_sur_class_2021_short)"  # Contextual controls
)

# ============================================================================
# Step 6: Fit models (4 variants)
# ============================================================================

models_dict = {}

model1 = mixedlm(formula, data=df_unstd_with, groups=df_unstd_with[random_intercept_var])
models_dict['unstd_with'] = model1.fit(reml=True)

model2 = mixedlm(formula, data=df_unstd_without, groups=df_unstd_without[random_intercept_var])
models_dict['unstd_without'] = model2.fit(reml=True)

model3 = mixedlm(formula, data=df_std_with, groups=df_std_with[random_intercept_var])
models_dict['std_with'] = model3.fit(reml=True)

model4 = mixedlm(formula, data=df_std_without, groups=df_std_without[random_intercept_var])
models_dict['std_without'] = model4.fit(reml=True)

# ============================================================================
# Step 7: Extract fixed effects
# ============================================================================

def extract_fixed_effects(result):
    fe_index = result.fe_params.index
    coefs = result.fe_params[fe_index].values
    ses = result.bse[fe_index].values
    zvals = result.tvalues[fe_index].values
    pvals = result.pvalues[fe_index].values
    ci_lower = coefs - 1.96 * ses
    ci_upper = coefs + 1.96 * ses

    return pd.DataFrame({
        'Variable': fe_index,
        'Coefficient': coefs,
        'Std. Error': ses,
        'z-value': zvals,
        'p-value': pvals,
        '95% CI Lower': ci_lower,
        '95% CI Upper': ci_upper
    }).round(6)

results_dict = {}
for key, result in models_dict.items():
    results_dict[key] = extract_fixed_effects(result)

output_dir = Path(r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2\LMM")
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# Step 8: Save results
# ============================================================================

# Unstandardized coefficients (original scale)
results_dict['unstd_with'].to_csv(
    output_dir / "1_DELTA_NDVI_unstd_WITH_outliers.csv", index=False
)

results_dict['unstd_without'].to_csv(
    output_dir / "2_DELTA_NDVI_unstd_WITHOUT_outliers.csv", index=False
)

# Standardized coefficients (1 SD change in X)
results_dict['std_with'].to_csv(
    output_dir / "3_DELTA_NDVI_std_WITH_outliers.csv", index=False
)

results_dict['std_without'].to_csv(
    output_dir / "4_DELTA_NDVI_std_WITHOUT_outliers.csv", index=False
)

# ============================================================================
# Step 9: Model fit comparison
# ============================================================================

model_fit_comparison = pd.DataFrame({
    'Model': ['WITH outliers', 'WITHOUT outliers'],
    'N Observations': [len(df_std_with), len(df_std_without)],
    'N Groups (NHDAs)': [df_std_with['nhda_id'].nunique(), df_std_without['nhda_id'].nunique()],
    'Log-Likelihood': [models_dict['std_with'].llf, models_dict['std_without'].llf],
    'AIC': [models_dict['std_with'].aic, models_dict['std_without'].aic],
    'BIC': [models_dict['std_with'].bic, models_dict['std_without'].bic]
})

model_fit_comparison.to_csv(
    output_dir / "5_DELTA_NDVI_model_fit.csv", index=False
)


# ============================================================================
# Step 11: Visualization with built-form emphasis
# ============================================================================

results_std_without = results_dict['std_without'].copy()

ordered_vars = (
    built_form_vars +
    temporal_control_vars +
    spatial_control_vars
)

plot_results = results_std_without[
    results_std_without['Variable'].isin(ordered_vars)
].copy()

plot_results['var_order'] = plot_results['Variable'].map(
    {var: idx for idx, var in enumerate(ordered_vars)}
)
plot_results = plot_results.sort_values('var_order').reset_index(drop=True)

# Clean variable labels
label_map = {
    'reldiff_built_up_ratio': 'Rel. $\Delta$ Built-up ratio',
    'reldiff_avg_building_height': 'Rel. $\Delta$ Building height',
    'reldiff_avg_building_footprint': 'Rel. $\Delta$ Building footprint',
    'diff_res_subclass_share_mfh_ab': '$\Delta$MFH-AB share',
    'years_since_construction_start': 'Years since construction',
    'lat': 'Latitude',
    'lon': 'Longitude'
}

plot_results['label'] = plot_results['Variable'].map(label_map)

# Role labels
def get_role(var):
    if var in built_form_vars:
        return 'Built-form characteristics'
    if var in temporal_control_vars:
        return 'Temporal control'
    if var in spatial_control_vars:
        return 'Spatial controls'
    return 'Other'

plot_results['role'] = plot_results['Variable'].apply(get_role)

# Same colors as LST model
role_colors = {
    'Built-form characteristics': '#ff595e',
    'Temporal control': '#ffca3a',
    'Spatial controls': '#b5a6c9'
}

plot_results['color'] = plot_results['role'].map(role_colors)

# Reverse order so first group appears at top
plot_results = plot_results.iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
y_pos = np.arange(len(plot_results))

# Confidence intervals and points
for i, row in plot_results.iterrows():

    ax.errorbar(
        row['Coefficient'],
        i,
        xerr=1.96 * row['Std. Error'],
        fmt='o',
        color=row['color'],
        ecolor=row['color'],
        alpha=0.7,
        markersize=9,
        markeredgecolor='black',
        markeredgewidth=0.8,
        elinewidth=2,
        capsize=4,
        zorder=3
    )

# Axis labels with significance stars below variable names
y_labels = []

for _, row in plot_results.iterrows():
    sig = ""
    if row['p-value'] < 0.001:
        sig = "\n$^{***}$"
    elif row['p-value'] < 0.01:
        sig = "\n$^{**}$"
    elif row['p-value'] < 0.05:
        sig = "\n$^{*}$"

    y_labels.append(row['label'] + sig)

ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel('Standardized coefficient', fontsize=10)

# Zero line
ax.axvline(0, color='black', linewidth=0.9, linestyle='--')

# Subtle grid
ax.grid(axis='x', alpha=0.25, linewidth=0.6)
ax.grid(axis='y', visible=False)

# Group separators
role_sequence = plot_results['role'].tolist()
for i in range(len(role_sequence) - 1):
    if role_sequence[i] != role_sequence[i + 1]:
        ax.axhline(i + 0.5, color='0.8', linewidth=0.7)

# Compact legend
from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Built-form',
           markerfacecolor=role_colors['Built-form characteristics'],
           markeredgecolor='black',
           alpha=0.7,
           markersize=10),

    Line2D([0], [0], marker='o', color='w',
           label='Temporal control',
           markerfacecolor=role_colors['Temporal control'],
           markeredgecolor='black',
           alpha=0.7,
           markersize=10),

    Line2D([0], [0], marker='o', color='w',
           label='Spatial controls',
           markerfacecolor=role_colors['Spatial controls'],
           markeredgecolor='black',
           alpha=0.7,
           markersize=10),
]

ax.legend(
    handles=legend_elements,
    loc='lower left',
    bbox_to_anchor=(1.02, 0),
    fontsize=8,
    frameon=False,
    title='Variable group',
    title_fontsize=9
)

# Clean title
ax.set_title(
    'Standardized coefficients for the ΔNDVI model',
    fontsize=11,
    fontweight='bold',
    pad=10
)

# Clean spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.subplots_adjust(right=0.78)

coef_plot = output_dir / "6_DELTA_NDVI_coefficient_plot_clean.jpg"
plt.savefig(coef_plot, dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# ============================================================================
# Step 12: Model diagnostics
# Residuals vs fitted values and normal Q-Q plot
# ============================================================================
print("\n" + "=" * 80)
print("STEP 12: MODEL DIAGNOSTICS")
print("=" * 80)

import statsmodels.api as sm

# Primary model: unstandardized model without outliers
diagnostic_model = models_dict['unstd_without']

# Extract conditional fitted values and residuals
fitted_values = np.asarray(diagnostic_model.fittedvalues, dtype=float)
residuals = np.asarray(diagnostic_model.resid, dtype=float)

# Remove non-finite values if present
valid_mask = np.isfinite(fitted_values) & np.isfinite(residuals)

fitted_values = fitted_values[valid_mask]
residuals = residuals[valid_mask]

# Standardized residuals
residual_mean = np.mean(residuals)
residual_sd = np.std(residuals, ddof=1)

if not np.isfinite(residual_sd) or residual_sd == 0:
    raise ValueError("Residual standard deviation is zero or non-finite.")

standardized_residuals = (
    residuals - residual_mean
) / residual_sd


# ============================================================================
# 1. Residuals versus fitted values
# ============================================================================

fig, ax = plt.subplots(figsize=(7.2, 5.2))

ax.scatter(
    fitted_values,
    standardized_residuals,
    s=18,
    alpha=0.25,
    edgecolors='none',
    color = "darkgrey"
)

# Horizontal zero line
ax.axhline(
    y=0,
    color='black',
    linestyle='--',
    linewidth=1
)

# LOWESS smoother
lowess_values = sm.nonparametric.lowess(
    endog=standardized_residuals,
    exog=fitted_values,
    frac=0.30,
    it=3,
    return_sorted=True
)

ax.plot(
    lowess_values[:, 0],
    lowess_values[:, 1],
    linewidth=2
)

ax.set_xlabel(
    r'Fitted $\Delta$NDVI values',
    fontsize=10
)

ax.set_ylabel(
    'Standardized residuals',
    fontsize=10
)

ax.set_title(
    r'Residuals versus fitted values for the $\Delta$NDVI LMM',
    fontsize=11,
    fontweight='bold',
    pad=10
)

ax.grid(
    axis='both',
    alpha=0.20,
    linewidth=0.6
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

residual_plot = (
    output_dir /
    "DELTA_NDVI_residuals_vs_fitted_without_outliers.jpg"
)

plt.savefig(
    residual_plot,
    dpi=300,
    bbox_inches='tight'
)

print(f"✓ Residual-versus-fitted plot saved: {residual_plot}")
plt.close()


# ============================================================================
# 2. Normal Q-Q plot
# ============================================================================

fig, ax = plt.subplots(figsize=(6.2, 5.2))

sm.qqplot(
    standardized_residuals,
    line='45',
    fit=True,
    ax=ax,
    marker='o',
    markerfacecolor='none',
    markeredgecolor='black',
    alpha=0.30
)

ax.set_xlabel(
    'Theoretical quantiles',
    fontsize=10
)

ax.set_ylabel(
    'Standardized residual quantiles',
    fontsize=10
)

ax.set_title(
    r'Normal Q--Q plot for the $\Delta$NDVI LMM',
    fontsize=11,
    fontweight='bold',
    pad=10
)

ax.grid(
    axis='both',
    alpha=0.20,
    linewidth=0.6
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

qq_plot = (
    output_dir /
    "DELTA_NDVI_normal_QQ_without_outliers.jpg"
)

plt.savefig(
    qq_plot,
    dpi=300,
    bbox_inches='tight'
)

print(f"✓ Normal Q-Q plot saved: {qq_plot}")
plt.close()